# WBF 앙상블 추론 전용 노트북 (Kaggle) — 체크포인트 Input → 제출 CSV

**목적**: 학습 노트북들이 만든 체크포인트를 Input으로 연결해 test셋을 WBF 앙상블 추론하고 제출 CSV를 만든다.
**학습은 하지 않는다.** 지금은 task2-masked 5-fold만으로 제출하고, task3/task4 체크포인트가 나오는 대로
**Input에 추가만 하면 자동으로 앙상블에 합류**한다.

| 모델 그룹 | 체크포인트 파일명 패턴 | 타입 |
|---|---|---|
| task2 | `medium_task2_syn74_masked_fold{0..4}_best.pth` | RF-DETR medium × 5 |
| task3 | `large_task3_full74_masked_ep*_last.pth` (누적 ep 최대인 것 1개 자동 선택) | RF-DETR large × 1 |
| task4 | `yolov8m_task4_syn74_masked_fold{0..4}_best.pt` | YOLOv8m × 최대 5 |

**동작 방식**
- 각 그룹의 체크포인트를 `/kaggle/input` 전체에서 **파일명으로 재귀 검색** → 어떤 슬러그의 Dataset/커밋 Output에
  들어 있어도 찾는다. 하나도 없는 그룹은 경고 후 제외 (task2만 연결하면 task2 단독 앙상블).
- 발견된 모든 모델(예: 5+1+5=11개)의 예측을 이미지 단위로 모아 **WBF 융합** → score ≥ 0.3 제출.
- 제출 파일명에 참여 그룹과 **weight**가 자동 반영: `submission_wbf_task2w1.csv`,
  `submission_wbf_task2w1+task3w2.5.csv` 등 → 조합/가중치 실험별로 스코어를 구분해 기록 가능.

**Input 연결 (필수 2 + 체크포인트)**
- competition 데이터의 `test_images`
- `task2_synthesized` — 라벨 매핑(1~74 ↔ 원본 category_id)의 신뢰 소스 (학습 노트북들과 동일한 방식)
- 체크포인트가 든 Dataset 또는 노트북 커밋 Output (여러 개 가능)

**참고 (WBF 투표 구조)**: WBF는 "모델 1개 = 1표"라서 그룹별 모델 수가 다르면(예: task2 5표 vs task3 1표)
표 수가 많은 그룹의 영향이 크다. 그룹별 가중치는 4번 셀 `MODEL_SPECS`의 `weight`로 조절할 수 있다 (기본 1.0).


In [ ]:
# [1. 환경 준비] 세 모델 타입을 모두 지원하도록 전부 설치합니다 (미참여 그룹이 있어도 무해).
#  ⚠ pip 설치는 세션 재시작 시 사라지므로 매 세션 이 셀부터 실행. Internet on 필요.
!pip install -q "rfdetr[train]" torchmetrics
!pip install -q ensemble-boxes
!pip install -q "ultralytics>=8.3"

import importlib
for m in ('rfdetr', 'torchmetrics', 'ensemble_boxes', 'ultralytics'):
    importlib.import_module(m)
    print('설치 확인 OK:', m)


In [ ]:
# [2. 저장소 준비] 팀 저장소를 clone하고 공통 코드(RF_DETR_split_ver)와 yejin 모듈을 import 경로에 추가합니다.
import os, sys, re, glob, json, shutil, math
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image

REPO_URL = 'https://github.com/kim-tae-yoon-0718/ai12-team01.git'
REPO_DIR = '/kaggle/working/ai12-team01'
REPO_BRANCH = 'main'   # ⚠ yejin 작업 브랜치가 main에 병합되기 전에 실행하려면 해당 브랜치명으로 변경
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
sys.path.insert(0, os.path.join(REPO_DIR, 'RF_DETR_split_ver'))
sys.path.insert(0, os.path.join(REPO_DIR, 'yejin'))

# ── 저장소 공통 코드에서 그대로 재사용하는 함수 ────────────────────────────────
from model import get_rfdetr_model                    # RF-DETR 생성 (checkpoint_path 로드)
from visualize import collect_predictions_ensemble    # RF-DETR 모델 리스트 추론 (저장소 함수)

# ── yejin/pipeline 공통 모듈 ──────────────────────────────────────────────────
from pipeline import cloud, wbf, viz
from pipeline.yolo import collect_predictions_ensemble_yolo   # YOLO 체크포인트 앙상블 수집


In [ ]:
# [3. 입력 경로 + 라벨 매핑]
#  라벨 매핑은 학습 노트북들과 동일하게 task2_synthesized의 pool JSON categories
#  (id 1~74, name=원본 category_id)를 신뢰 소스로 사용합니다.
TEST_IMG_DIR = cloud.find_input_dir('test_images')
POOL_DIR = cloud.find_input_dir('task2_synthesized')
assert TEST_IMG_DIR, 'test_images를 못 찾음 - competition 데이터 Input 연결 확인'
assert POOL_DIR, 'task2_synthesized를 못 찾음 - 라벨 매핑 소스로 필요 (Input 연결 확인)'

POOL_ANN_PATH = (glob.glob(os.path.join(POOL_DIR, 'annotations', '*.json')) or [None])[0]
with open(POOL_ANN_PATH, encoding='utf-8') as f:
    pool_coco = json.load(f)
cat2label = {int(c['name']): c['id'] for c in pool_coco['categories'] if c['id'] != 0}
label2cat = {v: k for k, v in cat2label.items()}
valid_labels = set(label2cat)
assert len(label2cat) == 74, f'라벨 매핑 {len(label2cat)}종 (기대: 74)'
print(f'TEST_IMG_DIR: {TEST_IMG_DIR} / 라벨 매핑 {len(label2cat)}종 로드 완료')


In [ ]:
# [4. 모델 그룹 레지스트리 + 체크포인트 자동 발견]
#  각 그룹의 체크포인트를 /kaggle/input 전체에서 파일명 패턴으로 재귀 검색합니다
#  (pipeline.cloud.find_files_under - 파일명 중복도 자동 제거).
#  -> 새 체크포인트가 나오면 그 Dataset/커밋 Output을 Input에 추가하기만 하면 됩니다 (코드 수정 불필요).
#  weight: WBF에서 이 그룹 소속 모델들의 투표 가중치 (기본 1.0, 그룹 간 균형 조절용)
MODEL_SPECS = [
    {'group': 'task2', 'type': 'rfdetr', 'variant': 'medium', 'weight': 1.0,
     'pattern': 'medium_task2_syn74_masked_fold*_best.pth'},
    {'group': 'task3', 'type': 'rfdetr', 'variant': 'large', 'weight': 1.0,
     'pattern': 'large_task3_full74_masked_ep*_last.pth', 'pick': 'max_ep'},
    {'group': 'task4', 'type': 'yolo', 'variant': None, 'weight': 1.0,
     'pattern': 'yolov8m_task4_syn74_masked_fold*_best.pt'},
]

COLLECT_SCORE_THR = 0.05   # 수집 threshold (WBF 전 단계 - 낮게 잡아 recall 확보)
WBF_IOU_THR       = 0.55   # WBF에서 같은 객체로 간주할 IoU
SUBMIT_SCORE_THR  = 0.3    # 제출 CSV 최소 confidence

def discover_checkpoints(spec):
    """spec의 파일명 패턴으로 /kaggle/input을 재귀 검색해 체크포인트 목록을 반환합니다."""
    hits = cloud.find_files_under('/kaggle/input', spec['pattern'])
    if spec.get('pick') == 'max_ep' and hits:
        # ep 번호가 가장 큰 파일 1개만 사용 (task3: 중간 세션 산출물이 섞여 있어도 최종본 선택)
        hits = [max(hits, key=lambda p: int(re.search(r'_ep(\d+)_', os.path.basename(p)).group(1)))]
    return hits

active_specs = []
for spec in MODEL_SPECS:
    ckpts = discover_checkpoints(spec)
    if ckpts:
        spec = dict(spec, checkpoints=ckpts)
        active_specs.append(spec)
        print(f"[{spec['group']}] {len(ckpts)}개 발견:")
        for p in ckpts:
            print('   ', p)
    else:
        print(f"[{spec['group']}] 체크포인트 없음 -> 이번 앙상블에서 제외 (패턴: {spec['pattern']})")

assert active_specs, '발견된 체크포인트가 하나도 없습니다 - Input 연결을 확인하세요'
n_models_total = sum(len(s['checkpoints']) for s in active_specs)
# 파일명에 그룹별 weight 포함: weight 실험끼리도 파일명으로 구분됩니다 (예: task2w1+task3w2.5)
ENSEMBLE_NAME = '+'.join(f"{s['group']}w{s['weight']:g}" for s in active_specs)
SUBMISSION_PATH = f'/kaggle/working/submission_wbf_{ENSEMBLE_NAME}.csv'
print(f'\n참여 그룹: {ENSEMBLE_NAME} / 총 모델 {n_models_total}개')
print('제출 파일:', SUBMISSION_PATH)


In [ ]:
# [5. test 예측 수집] 그룹별로 추론해 이미지 단위로 "전역 모델 인덱스" 기준으로 합칩니다.
#  - RF-DETR 그룹: 저장소 collect_predictions_ensemble() 그대로 사용
#  - YOLO 그룹: pipeline.yolo.collect_predictions_ensemble_yolo (cls 0-indexed -> 라벨 1~74 변환 포함)
#  - 라벨 정제(배경 0 등 제거): pipeline.wbf.filter_valid_labels

# 전역 구조: file_name -> {'image', 'by_model': {전역 모델 idx: (boxes, labels, scores)}}
merged = {}
model_weights = []   # 전역 모델 idx 순서의 WBF 가중치
offset = 0
for spec in active_specs:
    print(f"\n[{spec['group']}] 추론 시작 ({len(spec['checkpoints'])}개 모델)")
    if spec['type'] == 'rfdetr':
        models = [get_rfdetr_model(spec['variant'], checkpoint_path=p) for p in spec['checkpoints']]
        pred_data = collect_predictions_ensemble(models, TEST_IMG_DIR,
                                                 score_threshold=COLLECT_SCORE_THR)
        del models
        torch.cuda.empty_cache()
    else:
        pred_data = collect_predictions_ensemble_yolo(spec['checkpoints'], TEST_IMG_DIR,
                                                      conf_thr=COLLECT_SCORE_THR)

    wbf.filter_valid_labels(pred_data, valid_labels)

    for d in pred_data:
        boxes = d['pred_boxes'].numpy()
        labels = d['pred_labels'].numpy()
        scores = d['pred_scores'].numpy()
        fold_of = d['pred_fold'].numpy()

        m = merged.setdefault(d['file_name'], {'image': d['image'], 'by_model': {}})
        for local_idx in range(len(spec['checkpoints'])):
            sel = fold_of == local_idx
            m['by_model'][offset + local_idx] = (boxes[sel], labels[sel], scores[sel])

    model_weights.extend([spec['weight']] * len(spec['checkpoints']))
    offset += len(spec['checkpoints'])
    print(f"[{spec['group']}] 완료 - 누적 모델 {offset}개")

print(f'\ntest 이미지 {len(merged)}장 / 총 모델 {offset}개 / 합집합 박스 '
      f"{sum(len(v[0]) for m in merged.values() for v in m['by_model'].values())}개")


In [ ]:
# [6. WBF(Weighted Box Fusion) 융합] 전 그룹 모델을 한꺼번에 이미지 단위로 융합합니다.
#  모델 1개 = 1표(x 그룹 weight), conf_type='avg' 기준으로 동의 모델 수가 score에 반영됩니다.
#  융합 원리는 pipeline.wbf.fuse_merged_wbf docstring 참고.
fused_data = wbf.fuse_merged_wbf(merged, n_models_total, weights=model_weights,
                                 iou_thr=WBF_IOU_THR, skip_box_thr=COLLECT_SCORE_THR)
n_before = sum(len(v[0]) for m in merged.values() for v in m['by_model'].values())
n_after = sum(len(d['pred_boxes']) for d in fused_data)
print(f'융합 전 박스 {n_before}개 -> 융합 후 {n_after}개 (모델 {n_models_total}개, 그룹: {ENSEMBLE_NAME})')


In [ ]:
# [7. 추론 결과 클래스별 시각화] WBF 융합 예측을 클래스별 crop grid로 표시합니다
#  (pipeline.viz.show_pred_class_crops - 제출 기준과 동일한 threshold로 점검).
viz.show_pred_class_crops(fused_data, label2cat, score_thr=SUBMIT_SCORE_THR, max_per_class=8)


In [ ]:
# [8. 제출 CSV 생성] 파일명에 참여 그룹+weight가 반영됩니다 (pipeline.wbf.make_submission).
# 파일명 -> image_id 매핑을 눈으로 먼저 확인 (규칙이 다르면 pipeline.wbf.extract_image_id 수정)
for d in fused_data[:5]:
    print(d['file_name'], '->', wbf.extract_image_id(d['file_name']))

submission_df = wbf.make_submission(fused_data, label2cat, SUBMIT_SCORE_THR, SUBMISSION_PATH)
submission_df.head(10)


## 산출물 / 운영 방법

- `submission_wbf_{참여그룹+weight}.csv` — 커밋 후 Output에서 제출 (예: `submission_wbf_task2w1+task3w2.5.csv`)

**새 체크포인트 추가 절차 (코드 수정 없음)**
1. task3/task4 학습 커밋의 Output(또는 체크포인트 Dataset)을 이 노트북의 Input에 추가
2. Save & Run All → 4번 셀이 자동으로 발견해 앙상블에 포함 (출력에서 그룹/모델 수 확인)
3. 제출 파일명이 조합별로 달라지므로 스코어를 조합별로 비교 기록 가능

**조합 실험 팁**
- 특정 그룹을 일부러 빼고 싶으면 4번 셀 `MODEL_SPECS`에서 해당 항목을 주석 처리 (Input을 빼도 됨)
- 그룹 영향력 조절: `weight` 값 변경 (예: task3 단일 모델의 발언권을 키우려면 task3 weight를 2~3으로)
- WBF는 모델 1개=1표라 5-fold 그룹이 표 수에서 유리함 - 단일 모델 그룹(task3)의 기여가 스코어에서
  안 보이면 weight 조정 실험을 해볼 것
